# 🎙️➡️🎨 From Voice to Vision — Fase 1 & 2: Preprocessing + Feature

In questo notebook:
1. **Preprocessing** di una clip (trim silenzio + denoising) con confronto *prima/dopo* → copre i temi di corso *denoising* e *dominio della frequenza*.
2. **Estrazione feature** per tutte le 1440 clip: **log-Mel** (per la CNN) e **vettore MFCC** (per il ML classico), con **caching** su disco.
3. Figura riassuntiva: **spettrogramma medio per ogni emozione**.

In [ ]:
# === 1. Repo + librerie ===
REPO_URL = "https://github.com/<tuo-username>/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git","")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm
print("✓ pronto")

In [ ]:
# === 2. Indice dataset (scarica RAVDESS se serve) ===
from src import config, data_loader, preprocessing, features
data_loader.download_ravdess()
df = data_loader.build_index()
print("Clip:", len(df))

## Preprocessing: prima / dopo
Confrontiamo la clip grezza con la versione **trim + denoise**. Nota come il trim rimuove il
silenzio iniziale/finale e il denoising pulisce il rumore di fondo (RAVDESS è già pulito, quindi
l'effetto è lieve — ma è il tema di corso sul *denoising*).

In [ ]:
import librosa, librosa.display, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

row = df[df.emotion == "happy"].iloc[0]
raw = preprocessing.load_audio(row.path)
den = preprocessing.reduce_noise(raw)
proc = preprocessing.preprocess(row.path, denoise=True)   # trim + denoise + fix length

print("Durata grezza:", f"{len(raw)/config.SAMPLE_RATE:.2f}s",
      "| dopo fix_length:", f"{len(proc)/config.SAMPLE_RATE:.2f}s")
print("Ascolta — grezza vs pulita:")
display(Audio(raw, rate=config.SAMPLE_RATE)); display(Audio(den, rate=config.SAMPLE_RATE))

fig, ax = plt.subplots(2, 2, figsize=(13,7))
for j,(sig,ttl) in enumerate([(raw,"Grezza"),(den,"Denoised")]):
    librosa.display.waveshow(sig, sr=config.SAMPLE_RATE, ax=ax[0,j]); ax[0,j].set_title(f"Forma d'onda — {ttl}")
    S = librosa.power_to_db(librosa.feature.melspectrogram(
        y=sig, sr=config.SAMPLE_RATE, n_mels=config.N_MELS,
        n_fft=config.N_FFT, hop_length=config.HOP_LENGTH), ref=np.max)
    librosa.display.specshow(S, sr=config.SAMPLE_RATE, hop_length=config.HOP_LENGTH,
                             x_axis="time", y_axis="mel", ax=ax[1,j]); ax[1,j].set_title(f"log-Mel — {ttl}")
plt.tight_layout(); config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(config.FIGURES_DIR / "01_preprocessing_before_after.png", dpi=120); plt.show()

## Estrazione feature (tutte le clip)
Estraiamo **log-Mel** e **vettore MFCC** per ogni clip e le mettiamo in cache.
Usiamo `denoise=False` per il dataset principale (RAVDESS è pulito e il denoising rischia di
togliere informazione utile); il confronto denoise lo teniamo come esperimento per il report.
Richiede ~1-2 minuti.

In [ ]:
data = features.build_dataset(df, denoise=False, cache=True)
print("X_mel:", data["X_mel"].shape, "| X_vec:", data["X_vec"].shape,
      "| y:", data["y"].shape)
import numpy as np
for name in ["train","val","test"]:
    print(f"  {name:5s}: {(data['split']==name).sum()} clip")

## Firma spettrale di ogni emozione
Media dei log-Mel per classe: emozioni diverse hanno energia distribuita in modo diverso su
tempo e frequenza. È una figura d'effetto per il report (e motiva perché una CNN può distinguerle).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16,7))
for i, emo in enumerate(config.EMOTIONS):
    ax = axes[i//4, i%4]
    mask = (data["y"] == config.EMOTION_TO_ID[emo])
    mean_spec = data["X_mel"][mask].mean(0)
    librosa.display.specshow(mean_spec, sr=config.SAMPLE_RATE, hop_length=config.HOP_LENGTH,
                             x_axis="time", y_axis="mel", ax=ax)
    ax.set_title(f"{emo}  (n={mask.sum()})"); ax.label_outer()
plt.suptitle("Spettrogramma log-Mel medio per emozione (RAVDESS)", fontsize=14)
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "01_mean_spectrogram_per_emotion.png", dpi=120)
plt.show()

### ✅ Fase 1 & 2 completate
Abbiamo il dataset pronto come tensori (`X_mel`, `X_vec`) con le etichette e gli split.

**Mandami:** le shape stampate + le due figure (prima/dopo e spettrogrammi medi). Poi in **Fase 3** costruiamo i **modelli baseline** (SVM/RF sul vettore MFCC e la prima CNN sui log-Mel).